In [1]:
# =========================
# Cell 1 — Imports
# =========================
using LinearAlgebra
using Statistics
using Printf
using Parameters               # @with_kw
using Distributions            # Beta, etc.

In [2]:
# =========================
# Cell 2 — Structs (full model: separate blocks + shared type params)
# =========================

# --- Shared / cross-block primitives (objects used by both blocks) ---
@with_kw struct TypeParams
    # Worker type distribution x ~ Beta(a_x, b_x)
    a_x::Float64
    b_x::Float64

    # (Optional) Skilled firm type distribution y ~ Beta(a_y, b_y)
    # If you don't want y yet, keep defaults and ignore.
    a_y::Float64 = 2.0
    b_y::Float64 = 2.0
end

# --- Unskilled block parameters (only U-side primitives) ---
@with_kw struct UnskilledParams
    # Discounting / turnover
    r::Float64
    ν::Float64

    # Flow payoffs and shifter (for a fixed regime z you choose)
    bU::Float64
    bT::Float64
    PU::Float64

    # Matching
    μU::Float64
    ηU::Float64

    # Vacancy cost
    kU::Float64

    # Damage shocks: p' ~ Beta(αU,1)
    λU::Float64
    αU::Float64

    # Nash bargaining
    βU::Float64

    # Training completion
    ϕ::Float64

    # Training cost function c(x) = ac * exp(-x)
    ac::Float64
end

# --- Skilled block parameters (LMR-side primitives; placeholders you’ll use next) ---
@with_kw struct SkilledParams
    # Discounting / turnover (keep consistent with unskilled; still store here)
    r::Float64
    ν::Float64

    # Flow payoff in skilled unemployment
    bS::Float64

    # Output shifter
    PS::Float64

    # Matching in skilled market (Cobb–Douglas matching on effective searchers × vacancies)
    μS::Float64
    ηS::Float64

    # On-the-job search intensity
    s::Float64

    # Exogenous separation + job-type shocks
    ξS::Float64
    δS::Float64

    # Nash bargaining weight (hiring from unemployment and incremental-surplus split)
    βS::Float64

    # Vacancy posting cost function kS(y); simplest is constant kS
    kS_const::Float64

    # CES production f(x,y) = [ω x^ρ + (1-ω) y^ρ]^(1/ρ)
    ω::Float64
    ρ::Float64
end

# --- Grids (we’ll flesh out later; just define the containers now) ---
@with_kw struct UnskilledGrids
    x::Vector{Float64}
    p::Vector{Float64}
    x_edges::Vector{Float64}      # length Nx+1
end

@with_kw struct SkilledGrids
    x::Vector{Float64}
    y::Vector{Float64}
    x_edges::Vector{Float64}      # length Nx+1 (same construction as unskilled)
    y_edges::Vector{Float64}      # length Ny+1
end

# --- Solver settings (keep separate if you like; one for now) ---
@with_kw struct Settings
    maxit_inner::Int = 500
    maxit_outer::Int = 5_000
    tol_inner::Float64 = 1e-7
    tol_outer::Float64 = 1e-7
    verbose::Bool = true
end

Settings

In [3]:
# =========================
# Cell 3 — Utilities (shared + block-specific)
# =========================

# --- Robust bin edges from nodes on [0,1] ---
function bin_edges_from_nodes(grid::Vector{Float64})
    n = length(grid)
    @assert n >= 2
    @assert isapprox(grid[1], 0.0; atol=1e-12) && isapprox(grid[end], 1.0; atol=1e-12)
    @assert issorted(grid)

    e = zeros(Float64, n+1)
    e[1] = 0.0
    e[end] = 1.0
    @inbounds for i in 2:n
        e[i] = 0.5 * (grid[i-1] + grid[i])
    end
    return e
end

# --- Bin masses for Beta(a,b) on [0,1] given edges ---
function beta_bin_masses(edges::Vector{Float64}, a::Float64, b::Float64)
    dist = Beta(a, b)
    n = length(edges) - 1
    m = zeros(Float64, n)
    @inbounds for i in 1:n
        m[i] = cdf(dist, edges[i+1]) - cdf(dist, edges[i])
    end
    s = sum(m)
    @assert s > 0
    return m ./ s
end

# --- Unskilled matching ---
@inline qU(par::UnskilledParams, θ::Float64) = par.μU * θ^(-par.ηU)
@inline fU(par::UnskilledParams, θ::Float64) = θ * qU(par, θ)

# --- Unskilled training cost ---
@inline cU(par::UnskilledParams, x::Float64) = par.ac * exp(-x)

# --- Unskilled damage redraw CDF: p' ~ Beta(α,1) => G(p)=p^α ---
@inline function G_beta1(p::Float64, α::Float64)
    p <= 0.0 && return 0.0
    p >= 1.0 && return 1.0
    return p^α
end

# --- Skilled CES production f(x,y) ---
@inline function f_CES(x::Float64, y::Float64, ω::Float64, ρ::Float64)
    @assert 0.0 < ω < 1.0
    @assert ρ != 0.0
    return (ω * x^ρ + (1.0 - ω) * y^ρ)^(1.0 / ρ)
end

# --- Skilled vacancy cost (start with constant; later can generalize to kS(y)) ---
@inline kS(par::SkilledParams, y::Float64) = par.kS_const

kS (generic function with 1 method)

In [4]:
# =========================
# Cell 4 — State + Cache containers (keep unskilled as you wrote it; add skilled in parallel)
# =========================

# --- UNCHANGED (your unskilled state + cache) ---
@with_kw mutable struct UnskilledState
    # Outer objects (scalar tightness, vector pstar over x)
    θU::Float64                   # market tightness
    pstar::Vector{Float64}        # p*(x), length Nx
    τ::BitVector                  # τ(x) ∈ {0,1}, length Nx

    # Distributions (stored as bin masses over x; sum is total mass in that state)
    u_mass::Vector{Float64}       # mass in untrained unemployment by x-bin, length Nx
    t_mass::Vector{Float64}       # mass in training by x-bin, length Nx

    # Values on x-grid
    Usearch::Vector{Float64}      # U_U^{search}(x)
    U::Vector{Float64}            # U_U(x)
    Tval::Vector{Float64}         # T(x)

    # Match values on (x,p)
    EU::Matrix{Float64}           # E_U(x,p), size (Nx,Np)
    JU::Matrix{Float64}           # J_U(x,p), size (Nx,Np)

    # Convenience: frontier surplus at p=1
    J_frontier::Vector{Float64}   # J_U(x,1), length Nx
end

@with_kw mutable struct UnskilledCache
    Nx::Int
    Np::Int
    jp1::Int                      # index of p==1.0

    # Exogenous worker distribution as bin masses (robust, sums to 1)
    ell_mass::Vector{Float64}     # length Nx

    # Scratch
    tmp_x::Vector{Float64}        # length Nx
    tmp_p::Vector{Float64}        # length Np
end

# --- SKILLED: parallel organization ---
@with_kw mutable struct SkilledState
    # Outer scalar tightness object (LMR uses κ(z); we keep that)
    κ::Float64                    # κ (LMR tightness parameter)

    # Distributions / densities (bin masses over x, y; match density as matrix)
    uS_mass::Vector{Float64}      # skilled unemployment mass by x-bin, length Nx
    eS_mass::Vector{Float64}      # skilled employment mass by x-bin, length Nx
    mS_mass::Vector{Float64}      # trained population mass by x-bin, length Nx  (given by unskilled link in steady state)

    vS_mass::Vector{Float64}      # vacancy mass by y-bin, length Ny
    nS_mass::Vector{Float64}      # active posts mass by y-bin (filled+vacant), length Ny

    h_mass::Matrix{Float64}       # match mass by (x,y), size (Nx,Ny)

    # Core value/surplus object on (x,y)
    S::Matrix{Float64}            # surplus S(x,y), size (Nx,Ny)

    # Convenience
    feasible::BitMatrix           # S(x,y)≥0 indicator, size (Nx,Ny)
end

@with_kw mutable struct SkilledCache
    Nx::Int
    Ny::Int

    # Exogenous type distributions as bin masses
    ell_mass::Vector{Float64}     # x-bin masses (same as in unskilled cache), length Nx
    gamma_mass::Vector{Float64}   # y-bin masses, length Ny

    # Scratch buffers
    tmp_x::Vector{Float64}        # length Nx
    tmp_y::Vector{Float64}        # length Ny
    tmp_xy::Matrix{Float64}       # size (Nx,Ny)
end

SkilledCache

In [5]:
# =========================
# Cell 5 — Unskilled block: initialization (same representation as your notebook)
# =========================

# Find index of p==1.0 (frontier)
function frontier_index(p::Vector{Float64})
    j = findfirst(==(1.0), p)
    j === nothing && error("p-grid must contain 1.0 exactly (include 1.0 in range).")
    return j
end

"""
    initialize_unskilled(parU, types, gridU; θ0=1.0, pstar0=0.2, u_rate0=0.1)

Keeps your representation:
- ell_mass is computed robustly as x-bin masses from Beta(a_x,b_x)
- u_mass starts at u_rate0 * ell_mass; t_mass starts at 0
"""
function initialize_unskilled(par::UnskilledParams, types::TypeParams, grid::UnskilledGrids;
                             θ0::Real = 1.0,
                             pstar0::Real = 0.2,
                             u_rate0::Real = 0.1)

    Nx = length(grid.x)
    Np = length(grid.p)
    @assert length(grid.x_edges) == Nx + 1
    @assert isapprox(maximum(grid.p), 1.0; atol=1e-12) "p-grid must include 1.0"

    jp1 = frontier_index(grid.p)

    # Exogenous worker distribution: bin masses over x
    ell_mass = beta_bin_masses(grid.x_edges, types.a_x, types.b_x)  # sums to 1

    # Initial distributions: u integrates to u_rate0; t starts at 0
    u_mass = u_rate0 .* ell_mass
    t_mass = zeros(Nx)

    # Initial guesses
    θ_init = Float64(θ0)
    pstar_init = fill(Float64(pstar0), Nx)
    τ_init = falses(Nx)

    # Values
    Usearch = zeros(Nx)
    U       = zeros(Nx)
    Tval    = zeros(Nx)

    # Match values
    EU = zeros(Nx, Np)
    JU = zeros(Nx, Np)
    J_frontier = zeros(Nx)

    state = UnskilledState(;
        θU = θ_init,
        pstar = pstar_init,
        τ = τ_init,
        u_mass = u_mass,
        t_mass = t_mass,
        Usearch = Usearch,
        U = U,
        Tval = Tval,
        EU = EU,
        JU = JU,
        J_frontier = J_frontier
    )

    cache = UnskilledCache(;
        Nx = Nx,
        Np = Np,
        jp1 = jp1,
        ell_mass = ell_mass,
        tmp_x = zeros(Nx),
        tmp_p = zeros(Np)
    )

    return state, cache
end

initialize_unskilled

In [6]:
# =========================
# Cell 6 — Unskilled block: inner-loop utilities (Beta(α,1) quadrature on p-grid)
# =========================

function p_edges_from_nodes(p::Vector{Float64})
    n = length(p)
    @assert n >= 2
    @assert isapprox(p[1], 0.0; atol=1e-12) && isapprox(p[end], 1.0; atol=1e-12)
    @assert issorted(p)
    e = zeros(Float64, n+1)
    e[1] = 0.0
    e[end] = 1.0
    @inbounds for j in 2:n
        e[j] = 0.5 * (p[j-1] + p[j])
    end
    return e
end

# Bin masses under G(p)=p^α: m[j] = P(p' in (edge[j], edge[j+1]))
function G_bin_masses(p_edges::Vector{Float64}, α::Float64)
    n = length(p_edges) - 1
    m = zeros(Float64, n)
    @inbounds for j in 1:n
        lo = p_edges[j]
        hi = p_edges[j+1]
        m[j] = hi^α - lo^α
    end
    s = sum(m)
    @assert s > 0
    return m ./ s
end

# --- inner-loop updates ---
function update_T!(state::UnskilledState, par::UnskilledParams, grid::UnskilledGrids, US::Function)
    @inbounds for i in 1:length(grid.x)
        x = grid.x[i]
        state.Tval[i] = (par.bT + par.ϕ * US(x)) / (par.r + par.ϕ)
    end
    return nothing
end

function update_outside_and_tau!(state::UnskilledState, par::UnskilledParams, grid::UnskilledGrids)
    @inbounds for i in 1:length(grid.x)
        x = grid.x[i]
        Utrain = -cU(par, x) + state.Tval[i]
        if Utrain >= state.Usearch[i]
            state.U[i] = Utrain
            state.τ[i] = true
        else
            state.U[i] = state.Usearch[i]
            state.τ[i] = false
        end
    end
    return nothing
end

"""
Update (JU, EU) and J_frontier given outside value U(x) and pstar(x).

Uses your “surplus shortcut”:
  (r+ν+λ) S(x,p) = PU*x*p - rU(x) + λ ∫_{p*}^1 S(x,p') dG(p')   for p≥p*
and S=0 otherwise.
Then split: J=(1-β)S, E=U+βS.
"""
function update_match_values!(state::UnskilledState, par::UnskilledParams, grid::UnskilledGrids;
                             Gmass::Vector{Float64})

    Nx = length(grid.x)
    Np = length(grid.p)

    A  = par.r + par.ν + par.λU
    B  = par.λU

    @inbounds for i in 1:Nx
        x = grid.x[i]
        Ux = state.U[i]
        pstar = state.pstar[i]

        # Compute C(x)=∫_{p*}^1 (PU*x*p - rU) dG and Psurvive=∫_{p*}^1 dG
        Cx = 0.0
        Psurvive = 0.0
        for j in 1:Np
            if grid.p[j] >= pstar
                w = Gmass[j]
                Psurvive += w
                Cx += w * (par.PU * x * grid.p[j] - par.r * Ux)
            end
        end

        denom = 1.0 - (B / A) * Psurvive
        denom = max(denom, 1e-12)

        Mx = (Cx / A) / denom

        for j in 1:Np
            p = grid.p[j]
            if p >= pstar
                S = (par.PU * x * p - par.r * Ux + B * Mx) / A
                S = max(S, 0.0)
                state.JU[i, j] = (1.0 - par.βU) * S
                state.EU[i, j] = Ux + par.βU * S
            else
                state.JU[i, j] = 0.0
                state.EU[i, j] = Ux
            end
        end

        state.J_frontier[i] = state.JU[i, end]  # assumes p[end]=1.0
    end
    return nothing
end

function update_Usearch!(state::UnskilledState, par::UnskilledParams)
    θ = state.θU
    q = qU(par, θ)
    denom = par.r + par.ν + θ*q
    @inbounds for i in 1:length(state.U)
        Ux = state.U[i]
        J1 = state.J_frontier[i]
        state.Usearch[i] = (par.bU + θ*q*(Ux + (par.βU/(1.0-par.βU))*J1)) / denom
    end
    return nothing
end

update_Usearch! (generic function with 1 method)

In [7]:
# =========================
# Cell 7 — Unskilled block: inner-loop driver
# =========================

function solve_inner!(state::UnskilledState, cache::UnskilledCache,
                      par::UnskilledParams, grid::UnskilledGrids;
                      US::Function,
                      set::Settings,
                      Gmass::Vector{Float64})

    Usearch_old = similar(state.Usearch)
    U_old       = similar(state.U)

    for it in 1:set.maxit_inner
        copyto!(Usearch_old, state.Usearch)
        copyto!(U_old, state.U)

        update_T!(state, par, grid, US)
        update_outside_and_tau!(state, par, grid)
        update_match_values!(state, par, grid; Gmass=Gmass)
        update_Usearch!(state, par)

        d1 = maximum(abs.(state.Usearch .- Usearch_old))
        d2 = maximum(abs.(state.U .- U_old))
        d  = max(d1, d2)

        if set.verbose && (it == 1 || it % 25 == 0)
            @printf("inner it=%d  maxΔ=%.3e  (ΔUsearch=%.3e, ΔU=%.3e)\n", it, d, d1, d2)
        end
        if d < set.tol_inner
            set.verbose && println("inner converged in it=$it")
            return it
        end
    end

    set.verbose && println("inner hit maxit_inner=$(set.maxit_inner)")
    return set.maxit_inner
end

solve_inner! (generic function with 1 method)

In [8]:
# =========================
# Cell 8 — Unskilled block: outer loop (stationary dist + pstar + free entry) + solve_model!
# =========================

@inline function clamp_pstar(p; ϵ=1e-10)
    return min(1.0, max(ϵ, p))
end

function update_stationary_dist!(state::UnskilledState, cache::UnskilledCache,
                                 par::UnskilledParams)
    Nx = cache.Nx
    f  = fU(par, state.θU)

    @inbounds for i in 1:Nx
        ℓ = cache.ell_mass[i]
        τ = state.τ[i] ? 1.0 : 0.0
        Gp = G_beta1(state.pstar[i], par.αU)
        δ  = par.λU * Gp  # endogenous separation hazard

        # u[(f + τ + ν) + δ(1 + τ/ϕ)] = (ν + δ)ℓ
        denom = (f + τ + par.ν) + δ * (1.0 + τ / par.ϕ)
        denom = max(denom, 1e-14)

        u = ((par.ν + δ) * ℓ) / denom
        u = max(u, 0.0)

        state.u_mass[i] = u
        state.t_mass[i] = (τ / par.ϕ) * u
    end
    return nothing
end

function update_theta_free_entry!(state::UnskilledState,
                                  par::UnskilledParams; damp::Real=1.0)
    Jbar = sum(state.J_frontier .* state.u_mass)
    Jbar = max(Jbar, 1e-14)

    q_target = par.kU / Jbar
    q_target = max(q_target, 1e-14)

    θ_new = (par.μU / q_target)^(1.0 / par.ηU)
    state.θU = (1.0 - damp) * state.θU + damp * θ_new
    return θ_new
end

function update_pstar_from_J!(state::UnskilledState, cache::UnskilledCache,
                              grid::UnskilledGrids; damp::Real=1.0)

    Nx = cache.Nx
    Np = cache.Np

    pstar_old = similar(state.pstar)
    copyto!(pstar_old, state.pstar)

    @inbounds for i in 1:Nx
        if state.JU[i, 1] > 0.0
            p_new = 0.0
        elseif state.JU[i, end] <= 0.0
            p_new = 1.0
        else
            jpos = 0
            for j in 2:Np
                if state.JU[i, j] > 0.0
                    jpos = j
                    break
                end
            end
            if jpos == 0
                p_new = 1.0
            else
                j0 = jpos - 1
                p0, p1 = grid.p[j0], grid.p[jpos]
                J0, J1 = state.JU[i, j0], state.JU[i, jpos]
                denom = (J1 - J0)
                if abs(denom) < 1e-14
                    p_new = p0
                else
                    p_new = p0 + (-J0) * (p1 - p0) / denom
                end
            end
        end

        p_new = clamp_pstar(p_new)
        state.pstar[i] = (1.0 - damp) * state.pstar[i] + damp * p_new
    end

    return maximum(abs.(state.pstar .- pstar_old))
end

function solve_outer!(state::UnskilledState, cache::UnskilledCache,
                      par::UnskilledParams, grid::UnskilledGrids;
                      US::Function,
                      set::Settings,
                      Gmass::Vector{Float64},
                      damp_pstar::Real = 0.5,
                      damp_theta::Real = 0.5)

    pstar_old = similar(state.pstar)
    u_old     = similar(state.u_mass)

    for it in 1:set.maxit_outer
        θ_old = state.θU
        copyto!(pstar_old, state.pstar)
        copyto!(u_old, state.u_mass)

        solve_inner!(state, cache, par, grid; US=US, set=set, Gmass=Gmass)

        update_stationary_dist!(state, cache, par)
        Δp = update_pstar_from_J!(state, cache, grid; damp=damp_pstar)
        _θraw = update_theta_free_entry!(state, par; damp=damp_theta)

        Δθ = abs(state.θU - θ_old)
        Δu = maximum(abs.(state.u_mass .- u_old))
        maxΔ = max(Δθ, max(Δp, Δu))

        if set.verbose && (it == 1 || it % 10 == 0)
            @printf("outer it=%d  maxΔ=%.3e  (Δθ=%.3e, Δp*=%.3e, Δu=%.3e)  θ=%.6f\n",
                    it, maxΔ, Δθ, Δp, Δu, state.θU)
        end
        if maxΔ < set.tol_outer
            set.verbose && println("outer converged in it=$it")
            return it
        end
    end

    set.verbose && println("outer hit maxit_outer=$(set.maxit_outer)")
    return set.maxit_outer
end

function solve_model_unskilled!(state::UnskilledState, cache::UnskilledCache,
                                par::UnskilledParams, grid::UnskilledGrids;
                                US::Function,
                                set::Settings,
                                damp_pstar::Real = 0.5,
                                damp_theta::Real = 0.5)

    p_edges = p_edges_from_nodes(grid.p)
    Gmass   = G_bin_masses(p_edges, par.αU)

    return solve_outer!(state, cache, par, grid;
                        US=US, set=set, Gmass=Gmass,
                        damp_pstar=damp_pstar, damp_theta=damp_theta)
end

solve_model_unskilled! (generic function with 1 method)

In [9]:
# =============================================================================
# Skilled block (compatible with your existing TypeParams / SkilledParams /
# SkilledGrids / SkilledState / SkilledCache EXACTLY as defined above).
#
# Key change: no reliance on non-existent cache fields (perm_y, posted_y, S_old, ...).
# All extra scratch arrays are allocated ONCE inside solve_model_skilled! and reused.
# =============================================================================

@inline pos(x::Float64) = ifelse(x > 0.0, x, 0.0)

# --- Cobb–Douglas matching identity for κ ----------
# With M = μ Ũ^η V^(1-η),
# κ ≡ M/(Ũ V) = μ Ũ^(η-1) V^(-η)  ⇒  V = ( μ Ũ^(η-1) / κ )^(1/η).
@inline function implied_V_from_kappa(par::SkilledParams, Ueff::Float64, κ::Float64)
    κ    = max(κ, 1e-14)
    Ueff = max(Ueff, 1e-14)
    return (par.μS * Ueff^(par.ηS - 1.0) / κ)^(1.0 / par.ηS)
end

# =============================================================================
# Accounting: e_S(x)=∑_y h(x,y) and u_S(x)=m_S(x)-e_S(x)
# =============================================================================
function update_worker_accounting!(s::SkilledState, c::SkilledCache)
    Nx, Ny = c.Nx, c.Ny
    @inbounds for i in 1:Nx
        ei = 0.0
        for j in 1:Ny
            ei += s.h_mass[i, j]
        end
        s.eS_mass[i] = ei
        s.uS_mass[i] = max(s.mS_mass[i] - ei, 0.0)
    end
    return nothing
end

function update_feasible_mask!(s::SkilledState)
    @inbounds for i in axes(s.S, 1), j in axes(s.S, 2)
        s.feasible[i, j] = (s.S[i, j] >= 0.0)
    end
    return nothing
end

# =============================================================================
# Poaching externality term for all y:
#   P2(y) = ∬ ( S(x,y) - S(x,ỹ) )^+ h(x,ỹ) dx dỹ
#
# Implemented with per-x sorting (O(Nx*Ny log Ny)), no cache fields required.
# Scratch required: perm_y (Ny), S_sorted (Ny), h_sorted (Ny),
# and prefix sums over h and h*S_sorted.
# =============================================================================
function compute_P2_for_all_y!(P2::Vector{Float64},
                              s::SkilledState, c::SkilledCache,
                              perm_y::Vector{Int},
                              S_sorted::Vector{Float64},
                              h_sorted::Vector{Float64})
    Nx, Ny = c.Nx, c.Ny
    fill!(P2, 0.0)

    @inbounds for i in 1:Nx
        # initialize permutation
        for j in 1:Ny
            perm_y[j] = j
        end
        sort!(perm_y; by = j -> s.S[i, j])

        # collect sorted S and h in that order
        for r in 1:Ny
            j = perm_y[r]
            S_sorted[r] = s.S[i, j]
            h_sorted[r] = s.h_mass[i, j]
        end

        # prefix sums over h and h*S at strictly lower surplus
        prefH  = 0.0
        prefHS = 0.0

        for r in 1:Ny
            j  = perm_y[r]
            Sy = S_sorted[r]

            # ∑_{ỹ: S(x,ỹ)<S(x,y)} h(x,ỹ) * (Sy - S(x,ỹ))
            contrib = Sy * prefH - prefHS
            if contrib > 0.0
                P2[j] += contrib
            end

            hr = h_sorted[r]
            prefH  += hr
            prefHS += hr * Sy
        end
    end
    return nothing
end

# =============================================================================
# Free entry with endogenous posting
#   κ = min_y kS(y) / ((1-β) * gain(y))
# posted[y] = near-argmin within tolerance
#
# Scratch required: term1(Ny), P2(Ny), posted(Ny), plus sort buffers
# =============================================================================
function update_posted_set_and_kappa!(s::SkilledState, c::SkilledCache,
                                     par::SkilledParams, grid::SkilledGrids,
                                     term1::Vector{Float64},
                                     P2::Vector{Float64},
                                     posted::Vector{Bool},
                                     perm_y::Vector{Int},
                                     S_sorted::Vector{Float64},
                                     h_sorted::Vector{Float64};
                                     tol_post::Float64 = 1e-8)

    Nx, Ny = c.Nx, c.Ny
    β   = par.βS
    s_on = par.s

    # term1(y) = ∑_x S^+(x,y) * u(x)
    @inbounds for j in 1:Ny
        acc = 0.0
        for i in 1:Nx
            acc += pos(s.S[i, j]) * s.uS_mass[i]
        end
        term1[j] = acc
    end

    # P2(y)
    compute_P2_for_all_y!(P2, s, c, perm_y, S_sorted, h_sorted)

    # κ = min_y κ_req(y), κ_req(y) = kS(y) / ((1-β) * (term1(y) + s*P2(y)))
    κ_best = Inf
    @inbounds for j in 1:Ny
        gain = term1[j] + s_on * P2[j]
        if gain <= 1e-14
            continue
        end
        y = grid.y[j]
        κ_req = kS(par, y) / ((1.0 - β) * gain)
        if κ_req < κ_best
            κ_best = κ_req
        end
    end
    κ_best = max(κ_best, 1e-12)

    # posted set = { y : κ_req(y) ≈ κ_best }
    @inbounds for j in 1:Ny
        gain = term1[j] + s_on * P2[j]
        if gain <= 1e-14
            posted[j] = false
            continue
        end
        y = grid.y[j]
        κ_req = kS(par, y) / ((1.0 - β) * gain)
        posted[j] = (abs(κ_req - κ_best) <= tol_post * max(1.0, κ_best))
    end

    if !isfinite(κ_best)
        # No y had gain>eps. Economically: no profitable posting.
        # Numerically: keep κ finite and post nothing.
        fill!(posted, false)
        return 1e12  # BIG κ => tiny V (or return previous κ outside)
    end

    return κ_best
end

# =============================================================================
# Vacancy distribution given κ and posted set:
#   - Ueff = sum(u) + s*sum(e)
#   - Vtot implied by κ from matching identity
#   - allocate Vtot across posted y proportional to gamma_mass
# =============================================================================
function update_vacancies_given_kappa_and_posted!(s::SkilledState, c::SkilledCache,
                                                 par::SkilledParams,
                                                 posted::Vector{Bool})

    Ny = c.Ny

    Ueff = sum(s.uS_mass) + par.s * sum(s.eS_mass)
    Vtot = implied_V_from_kappa(par, Ueff, s.κ)

    mass_post = 0.0
    @inbounds for j in 1:Ny
        mass_post += posted[j] ? c.gamma_mass[j] : 0.0
    end

    if mass_post <= 1e-14
        # fallback: post everywhere
        @inbounds for j in 1:Ny
            posted[j] = true
        end
        mass_post = max(sum(c.gamma_mass), 1e-14)
    else
        mass_post = max(mass_post, 1e-14)
    end

    @inbounds for j in 1:Ny
        s.vS_mass[j] = posted[j] ? (Vtot * c.gamma_mass[j] / mass_post) : 0.0
    end

    return Vtot
end

# =============================================================================
# One Bellman update for surplus S.
#
# Scratch required: Ax(Nx), Dx(Nx), By(Ny), P2(Ny) + sort buffers.
# =============================================================================
function update_surplus_candidate!(S_candidate::Matrix{Float64},
                                  s::SkilledState, c::SkilledCache,
                                  par::SkilledParams, grid::SkilledGrids,
                                  Ax::Vector{Float64},
                                  Dx::Vector{Float64},
                                  By::Vector{Float64},
                                  P2::Vector{Float64},
                                  perm_y::Vector{Int},
                                  S_sorted::Vector{Float64},
                                  h_sorted::Vector{Float64})

    Nx, Ny = c.Nx, c.Ny
    rbar = par.r + par.ν + par.ξS + par.δS
    κ, β, s_on = s.κ, par.βS, par.s

    # Ax(x) = ∫ S^+(x,ỹ) v(ỹ) dỹ  (implemented as sum over vS_mass)
    # Dx(x) = ∫ S^+(x,y') γ(y') dy' (implemented as sum over gamma_mass)
    @inbounds for i in 1:Nx
        a = 0.0
        d = 0.0
        for j in 1:Ny
            sij = pos(s.S[i, j])
            a += sij * s.vS_mass[j]
            d += sij * c.gamma_mass[j]
        end
        Ax[i] = a
        Dx[i] = d
    end

    # By(y) = ∫ S^+(x̃,y) u(x̃) dx̃
    @inbounds for j in 1:Ny
        b = 0.0
        for i in 1:Nx
            b += pos(s.S[i, j]) * s.uS_mass[i]
        end
        By[j] = b
    end

    # P2(y)
    compute_P2_for_all_y!(P2, s, c, perm_y, S_sorted, h_sorted)

    inv_rbar = 1.0 / max(rbar, 1e-14)

    @inbounds for i in 1:Nx
        x = grid.x[i]

        term_A = -κ * β * Ax[i]
        term_D = par.δS * Dx[i]

        for j in 1:Ny
            y = grid.y[j]
            flow = par.PS * f_CES(x, y, par.ω, par.ρ) - par.bS + kS(par, y)

            term_B  = -κ * (1.0 - β) * By[j]
            term_P2 = -s_on * κ * (1.0 - β) * P2[j]

            # + sκβ ∫ (S(x,ỹ)-S(x,y))^+ v(ỹ) dỹ
            Sxy = s.S[i, j]
            inc = 0.0
            for jp in 1:Ny
                inc += s.vS_mass[jp] * pos(s.S[i, jp] - Sxy)
            end
            term_inc = s_on * κ * β * inc

            S_candidate[i, j] = (flow + term_A + term_B + term_P2 + term_inc + term_D) * inv_rbar
        end
    end

    return nothing
end

# =============================================================================
# Inner loop: solve surplus fixed point with damping.
#
# Scratch required: Sold(Nx,Ny), Snew(Nx,Ny) already allocated by caller.
# =============================================================================
function solve_surplus_inner!(s::SkilledState, c::SkilledCache,
                             par::SkilledParams, grid::SkilledGrids;
                             set::Settings,
                             Sold::Matrix{Float64},
                             Snew::Matrix{Float64},
                             Ax::Vector{Float64},
                             Dx::Vector{Float64},
                             By::Vector{Float64},
                             P2::Vector{Float64},
                             perm_y::Vector{Int},
                             S_sorted::Vector{Float64},
                             h_sorted::Vector{Float64},
                             damp_S::Float64 = 0.5)

    for it in 1:set.maxit_inner
        copyto!(Sold, s.S)

        update_surplus_candidate!(Snew, s, c, par, grid, Ax, Dx, By, P2, perm_y, S_sorted, h_sorted)

        @inbounds for i in axes(s.S, 1), j in axes(s.S, 2)
            s.S[i, j] = (1.0 - damp_S) * Sold[i, j] + damp_S * Snew[i, j]
        end
        update_feasible_mask!(s)

        dS = 0.0
        @inbounds for i in axes(s.S, 1), j in axes(s.S, 2)
            dS = max(dS, abs(s.S[i, j] - Sold[i, j]))
        end

        if set.verbose && (it == 1 || it % 25 == 0)
            @printf("skilled inner it=%d  maxΔS=%.3e\n", it, dS)
        end
        if dS < set.tol_inner
            set.verbose && println("skilled inner converged in it=$it")
            return it
        end
    end

    set.verbose && println("skilled inner hit maxit_inner=$(set.maxit_inner)")
    return set.maxit_inner
end

# =============================================================================
# Stationary h update: LMR Eq (14)
#
# Scratch required: perm_y(Ny), v_sorted(Ny), sufV(Ny+1), hnew(Nx,Ny)
# =============================================================================
function update_h_stationary!(s::SkilledState, c::SkilledCache,
                             par::SkilledParams,
                             perm_y::Vector{Int},
                             v_sorted::Vector{Float64},
                             sufV::Vector{Float64},
                             hnew::Matrix{Float64})

    Nx, Ny = c.Nx, c.Ny
    ν, ξ, δ = par.ν, par.ξS, par.δS
    κ, s_on = s.κ, par.s

    # refresh e(x) from current h (needed for inflow_shock)
    @inbounds for i in 1:Nx
        ei = 0.0
        for j in 1:Ny
            ei += s.h_mass[i, j]
        end
        s.eS_mass[i] = ei
    end

    fill!(hnew, 0.0)

    @inbounds for i in 1:Nx
        # sort y by increasing S(x,·)
        for j in 1:Ny
            perm_y[j] = j
        end
        sort!(perm_y; by = j -> s.S[i, j])

        # v_sorted[r] = v(perm_y[r])
        for r in 1:Ny
            v_sorted[r] = s.vS_mass[perm_y[r]]
        end

        # sufV[r] = ∑_{q=r..Ny} v_sorted[q]
        sufV[Ny + 1] = 0.0
        for r in Ny:-1:1
            sufV[r] = sufV[r + 1] + v_sorted[r]
        end

        prefH = 0.0  # mass in strictly lower-surplus feasible jobs (in this x-row)

        for r in 1:Ny
            j = perm_y[r]
            Sij = s.S[i, j]

            if Sij < 0.0
                hnew[i, j] = 0.0
                continue
            end

            vBbar = sufV[r]
            denom = max(ν + ξ + δ + s_on * κ * vBbar, 1e-14)

            inflow_match = κ * s.vS_mass[j] * (s.uS_mass[i] + s_on * prefH)
            inflow_shock = δ * c.gamma_mass[j] * s.eS_mass[i]

            hij = (inflow_match + inflow_shock) / denom
            hij = max(hij, 0.0)

            hnew[i, j] = hij
            prefH += hij
        end
    end

    copyto!(s.h_mass, hnew)
    return nothing
end

# =============================================================================
# Skilled unemployment value:
#   (r+ν) U_S(x) = bS + κ ∫ βS S^+(x,y) v(y) dy
# =============================================================================
function update_US_from_surplus!(US::Vector{Float64},
                                s::SkilledState, c::SkilledCache,
                                par::SkilledParams)

    Nx, Ny = c.Nx, c.Ny
    denom = max(par.r + par.ν, 1e-14)

    @inbounds for i in 1:Nx
        gain = 0.0
        for j in 1:Ny
            gain += pos(s.S[i, j]) * s.vS_mass[j]
        end
        US[i] = (par.bS + s.κ * par.βS * gain) / denom
    end
    return nothing
end

# =============================================================================
# Outer loop (skilled block)
# =============================================================================
function solve_model_skilled_old!(s::SkilledState, c::SkilledCache,
                             par::SkilledParams, grid::SkilledGrids;
                             set::Settings,
                             US_out::Vector{Float64},
                             damp_h::Float64 = 0.3,
                             damp_κ::Float64 = 0.3,
                             damp_S::Float64 = 0.3,
                             tol_post::Float64 = 1e-8)

    Nx, Ny = c.Nx, c.Ny

    # -------------------------------------------------------------------------
    # Local scratch (allocated once; replaces the missing cache fields)
    # -------------------------------------------------------------------------
    posted = Vector{Bool}(undef, Ny)
    fill!(posted, false)
    term1    = zeros(Float64, Ny)
    P2       = zeros(Float64, Ny)

    perm_y   = collect(1:Ny)
    S_sorted = zeros(Float64, Ny)
    h_sorted = zeros(Float64, Ny)

    Ax       = zeros(Float64, Nx)
    Dx       = zeros(Float64, Nx)
    By       = zeros(Float64, Ny)

    Sold_in  = similar(s.S)
    Snew_in  = similar(s.S)

    hnew     = similar(s.h_mass)

    v_sorted = zeros(Float64, Ny)
    sufV     = zeros(Float64, Ny + 1)

    S_old_outer = similar(s.S)
    h_old_outer = similar(s.h_mass)
    u_old_outer = similar(s.uS_mass)

    # -------------------------------------------------------------------------
    # Iterate
    # -------------------------------------------------------------------------
    for it in 1:set.maxit_outer
        κ_old = s.κ
        copyto!(S_old_outer, s.S)
        copyto!(h_old_outer, s.h_mass)
        copyto!(u_old_outer, s.uS_mass)

        # (1) accounting u,e from m and current h
        update_worker_accounting!(s, c)

        # (2) free entry ⇒ κ_fe and posted set
        κ_fe = update_posted_set_and_kappa!(s, c, par, grid,
                                            term1, P2, posted,
                                            perm_y, S_sorted, h_sorted;
                                            tol_post=tol_post)

        # damp κ and build vacancies from κ + posted set
        s.κ = (1.0 - damp_κ) * s.κ + damp_κ * κ_fe
        s.κ = clamp(s.κ, 1e-4, 50.0)
        update_vacancies_given_kappa_and_posted!(s, c, par, posted)

        # (3) surplus fixed point given κ,u,v,h
        solve_surplus_inner!(s, c, par, grid; set=set,
                             Sold=Sold_in, Snew=Snew_in,
                             Ax=Ax, Dx=Dx, By=By, P2=P2,
                             perm_y=perm_y, S_sorted=S_sorted, h_sorted=h_sorted,
                             damp_S=damp_S)

        # (4) stationary h (LMR Eq 14), damped
        update_h_stationary!(s, c, par, perm_y, v_sorted, sufV, hnew)
        @inbounds for i in axes(s.h_mass, 1), j in axes(s.h_mass, 2)
            s.h_mass[i, j] = (1.0 - damp_h) * h_old_outer[i, j] + damp_h * s.h_mass[i, j]
        end

        # refresh accounting after damped h
        update_worker_accounting!(s, c)

        # (5) U_S(x)
        update_US_from_surplus!(US_out, s, c, par)

        # diagnostics
        dκ = abs(s.κ - κ_old)

        dS = 0.0
        @inbounds for i in axes(s.S, 1), j in axes(s.S, 2)
            dS = max(dS, abs(s.S[i, j] - S_old_outer[i, j]))
        end

        dh = 0.0
        @inbounds for i in axes(s.h_mass, 1), j in axes(s.h_mass, 2)
            dh = max(dh, abs(s.h_mass[i, j] - h_old_outer[i, j]))
        end

        du = 0.0
        @inbounds for i in eachindex(s.uS_mass)
            du = max(du, abs(s.uS_mass[i] - u_old_outer[i]))
        end

        maxΔ = max(dκ, max(dS, max(dh, du)))

        if set.verbose && (it == 1 || it % 10 == 0)
            @printf("skilled outer it=%d  maxΔ=%.3e  (Δκ=%.3e, ΔS=%.3e, Δh=%.3e, Δu=%.3e)  κ=%.6f\n",
                    it, maxΔ, dκ, dS, dh, du, s.κ)
        end

        if maxΔ < set.tol_outer
            set.verbose && println("skilled outer converged in it=$it")
            return it
        end
    end

    set.verbose && println("skilled outer hit maxit_outer=$(set.maxit_outer)")
    return set.maxit_outer
end

solve_model_skilled_old! (generic function with 1 method)

In [10]:
function solve_model_skilled!(s::SkilledState, c::SkilledCache,
                             par::SkilledParams, grid::SkilledGrids;
                             set::Settings,
                             US_out::Vector{Float64},
                             damp_h::Float64 = 0.3,
                             damp_κ::Float64 = 0.3,
                             damp_S::Float64 = 0.3,
                             tol_post::Float64 = 1e-8)

    Nx, Ny = c.Nx, c.Ny

    # -------------------------------------------------------------------------
    # Local scratch (allocated once; replaces the missing cache fields)
    # -------------------------------------------------------------------------
    posted = Vector{Bool}(undef, Ny)
    fill!(posted, false)
    term1    = zeros(Float64, Ny)
    P2       = zeros(Float64, Ny)

    perm_y   = collect(1:Ny)
    S_sorted = zeros(Float64, Ny)
    h_sorted = zeros(Float64, Ny)

    Ax       = zeros(Float64, Nx)
    Dx       = zeros(Float64, Nx)
    By       = zeros(Float64, Ny)

    Sold_in  = similar(s.S)
    Snew_in  = similar(s.S)

    hnew     = similar(s.h_mass)

    v_sorted = zeros(Float64, Ny)
    sufV     = zeros(Float64, Ny + 1)

    S_old_outer = similar(s.S)
    h_old_outer = similar(s.h_mass)
    u_old_outer = similar(s.uS_mass)

    # -------------------------------------------------------------------------
    # Iterate
    # -------------------------------------------------------------------------
    for it in 1:set.maxit_outer
        κ_old = s.κ
        copyto!(S_old_outer, s.S)
        copyto!(h_old_outer, s.h_mass)
        copyto!(u_old_outer, s.uS_mass)

        # (1) accounting u,e from m and current h
        update_worker_accounting!(s, c)

        # (2) ensure we have v consistent with CURRENT κ (and some posted set)
        #     We need a posted set to allocate V across y. Use current S as proxy.
        begin
            _ = update_posted_set_and_kappa!(s, c, par, grid,
                                            term1, P2, posted,
                                            perm_y, S_sorted, h_sorted;
                                            tol_post=tol_post)
            # keep κ as-is (do NOT use '_' here), just allocate vacancies
            update_vacancies_given_kappa_and_posted!(s, c, par, posted)
        end

        # (3) surplus fixed point given κ,u,v,h  (κ is the "state" here)
        solve_surplus_inner!(s, c, par, grid; set=set,
                             Sold=Sold_in, Snew=Snew_in,
                             Ax=Ax, Dx=Dx, By=By, P2=P2,
                             perm_y=perm_y, S_sorted=S_sorted, h_sorted=h_sorted,
                             damp_S=damp_S)

        # (4) NOW free entry ⇒ κ_fe and posted set computed from UPDATED S
        κ_fe = update_posted_set_and_kappa!(s, c, par, grid,
                                            term1, P2, posted,
                                            perm_y, S_sorted, h_sorted;
                                            tol_post=tol_post)

        # damp κ and rebuild vacancies from NEW κ + posted set
        s.κ = (1.0 - damp_κ) * s.κ + damp_κ * κ_fe
        s.κ = clamp(s.κ, 1e-4, 50.0)
        update_vacancies_given_kappa_and_posted!(s, c, par, posted)

        # (5) stationary h (LMR Eq 14), then damp h
        update_h_stationary!(s, c, par, perm_y, v_sorted, sufV, hnew)
        @inbounds for i in axes(s.h_mass, 1), j in axes(s.h_mass, 2)
            s.h_mass[i, j] = (1.0 - damp_h) * h_old_outer[i, j] + damp_h * s.h_mass[i, j]
        end

        # refresh accounting after damped h
        update_worker_accounting!(s, c)

        # (6) U_S(x)
        update_US_from_surplus!(US_out, s, c, par)

        # diagnostics
        dκ = abs(s.κ - κ_old)

        dS = 0.0
        @inbounds for i in axes(s.S, 1), j in axes(s.S, 2)
            dS = max(dS, abs(s.S[i, j] - S_old_outer[i, j]))
        end

        dh = 0.0
        @inbounds for i in axes(s.h_mass, 1), j in axes(s.h_mass, 2)
            dh = max(dh, abs(s.h_mass[i, j] - h_old_outer[i, j]))
        end

        du = 0.0
        @inbounds for i in eachindex(s.uS_mass)
            du = max(du, abs(s.uS_mass[i] - u_old_outer[i]))
        end

        maxΔ = max(dκ, max(dS, max(dh, du)))

        if set.verbose && (it == 1 || it % 10 == 0)
            @printf("skilled outer it=%d  maxΔ=%.3e  (Δκ=%.3e, ΔS=%.3e, Δh=%.3e, Δu=%.3e)  κ=%.6f\n",
                    it, maxΔ, dκ, dS, dh, du, s.κ)
            @printf("              posted share = %.3f\n", mean(posted))
        end

        if maxΔ < set.tol_outer
            set.verbose && println("skilled outer converged in it=$it")
            return it
        end
    end

    set.verbose && println("skilled outer hit maxit_outer=$(set.maxit_outer)")
    return set.maxit_outer
end

solve_model_skilled! (generic function with 1 method)

In [11]:
# =========================
# Cell — Full model solver (iterate unskilled <-> skilled until cross-block consistency)
# =========================

# Piecewise-constant US(x) from bin edges (consistent with your "bin mass" representation)
function make_US_function(gridU::UnskilledGrids, US_vec::Vector{Float64})
    edges = gridU.x_edges
    @assert length(US_vec) == length(gridU.x)
    return (x::Float64) -> begin
        if x <= edges[1] + 1e-15
            return US_vec[1]
        elseif x >= edges[end] - 1e-15
            return US_vec[end]
        else
            i = searchsortedlast(edges, x)         # gives bin index in 1..Nx
            i = min(max(i, 1), length(US_vec))
            return US_vec[i]
        end
    end
end

function solve_full_model!(u_state::UnskilledState, u_cache::UnskilledCache,
                           s_state::SkilledState,   s_cache::SkilledCache,
                           parU::UnskilledParams,   gridU::UnskilledGrids,
                           parS::SkilledParams,     gridS::SkilledGrids;
                           set::Settings,
                           maxit_global::Int = 500,
                           tol_global::Float64 = 1e-7,
                           damp_mS::Real = 0.9,
                           damp_US::Real = 0.9)

    Nx = u_cache.Nx
    US_vec = zeros(Float64, Nx)          # skilled unemployment values on x-grid
    US_old = similar(US_vec)

    # start with a flat guess (can be overwritten by current s_state if you prefer)
    fill!(US_vec, parS.bS / max(parS.r + parS.ν, 1e-14))

    for it in 1:maxit_global
        copyto!(US_old, US_vec)

        # --- Unskilled block given US(x) ---
        USfun = make_US_function(gridU, US_vec)
        solve_model_unskilled!(u_state, u_cache, parU, gridU; US=USfun, set=set)

        # implied trained mass in steady state: mS = (ϕ/ν) t   (bin masses)
        mS_new = (parU.ϕ / parU.ν) .* u_state.t_mass
        @. s_state.mS_mass = (1.0 - damp_mS) * s_state.mS_mass + damp_mS * mS_new

        # --- Skilled block given mS_mass ---
        solve_model_skilled!(s_state, s_cache, parS, gridS; set=set, US_out=US_vec,
        damp_h = 0.50,
        damp_κ = 0.30,
        damp_S = 0.30,)

        # damp US update (optional extra smoothing)
        @. US_vec = (1.0 - damp_US) * US_old + damp_US * US_vec

        dUS = maximum(abs.(US_vec .- US_old))
        dmS = maximum(abs.(s_state.mS_mass .- mS_new))  # “distance to implied” after damping, still informative
        maxΔ = max(dUS, dmS)

        if set.verbose && (it == 1 || it % 10 == 0)
            @printf("GLOBAL it=%d  maxΔ=%.3e  (ΔUS=%.3e, ΔmS=%.3e)\n", it, maxΔ, dUS, dmS)
        end
        if maxΔ < tol_global
            set.verbose && println("GLOBAL converged in it=$it")
            return it
        end
    end

    set.verbose && println("GLOBAL hit maxit_global=$maxit_global")
    return maxit_global
end

solve_full_model! (generic function with 1 method)

In [12]:
# =========================
# Cell — initialize_model (build grids, caches, states for BOTH blocks)
# =========================

function initialize_skilled(parS::SkilledParams, types::TypeParams, gridS::SkilledGrids;
                            κ0::Float64 = 0.2,
                            mS_rate0::Float64 = 0.10)

    Nx = length(gridS.x)
    Ny = length(gridS.y)

    # -------------------------------------------------------------------------
    # Bin masses from Beta distributions on [0,1] using the edges you already store
    #   ell_mass   = worker type masses over x-bins
    #   gamma_mass = firm type masses over y-bins
    # -------------------------------------------------------------------------
    ell_mass   = beta_bin_masses(gridS.x_edges, types.a_x, types.b_x)          # length Nx
    gamma_mass = beta_bin_masses(gridS.y_edges, types.a_y, types.b_y)          # length Ny

    # Safety normalization (beta_bin_masses already normalizes, but keep robust)
    ell_mass   ./= max(sum(ell_mass), 1e-14)
    gamma_mass ./= max(sum(gamma_mass), 1e-14)

    # ---- cache (MUST fill all fields in your SkilledCache struct) ----
    cache = SkilledCache(
        Nx = Nx,
        Ny = Ny,
        ell_mass = ell_mass,
        gamma_mass = gamma_mass,
        tmp_x = zeros(Float64, Nx),
        tmp_y = zeros(Float64, Ny),
        tmp_xy = zeros(Float64, Nx, Ny)
    )

    # ---- state (MUST fill all fields in your SkilledState struct) ----
    state = SkilledState(
        κ = κ0,

        # trained mass mS(x): BIN MASS (includes Δx implicitly through binning)
        mS_mass = mS_rate0 .* copy(ell_mass),
        uS_mass = zeros(Float64, Nx),
        eS_mass = zeros(Float64, Nx),

        vS_mass = zeros(Float64, Ny),
        nS_mass = zeros(Float64, Ny),

        h_mass  = zeros(Float64, Nx, Ny),

        S = zeros(Float64, Nx, Ny),
        feasible = falses(Nx, Ny)
    )

    # -------------------------------------------------------------------------
    # Initial accounting: start with no matches => e=0, u=mS
    # -------------------------------------------------------------------------
    @inbounds for i in 1:Nx
        state.eS_mass[i] = 0.0
        state.uS_mass[i] = state.mS_mass[i]
    end
    fill!(state.h_mass, 0.0)

    # -------------------------------------------------------------------------
    # Initialize vacancies from κ0, posting everywhere (no posted set needed here)
    # v(y) = Vtot * gamma_mass(y)
    # -------------------------------------------------------------------------
    Ueff = sum(state.uS_mass) + parS.s * sum(state.eS_mass)
    Vtot = implied_V_from_kappa(parS, Ueff, state.κ)

    @inbounds for j in 1:Ny
        state.vS_mass[j] = Vtot * cache.gamma_mass[j]
        state.nS_mass[j] = state.vS_mass[j]  # simple consistent default: active posts = vacancies initially
    end

    # -------------------------------------------------------------------------
    # Initial surplus guess:
    #   S0(x,y) = (PS f(x,y) - bS + kS(y)) / (r+ν+ξ+δ)
    # -------------------------------------------------------------------------
    rbar = parS.r + parS.ν + parS.ξS + parS.δS
    denom = max(rbar, 1e-14)

    @inbounds for i in 1:Nx
        x = gridS.x[i]
        for j in 1:Ny
            y = gridS.y[j]
            flow = parS.PS * f_CES(x, y, parS.ω, parS.ρ) - parS.bS + kS(parS, y)
            Sij = flow / denom
            state.S[i, j] = Sij
            state.feasible[i, j] = (Sij >= 0.0)
        end
    end

    return state, cache
end

"""
initialize_model(; grid sizes + parameter structs)

Returns:
  (gridU, gridS, u_state, u_cache, s_state, s_cache)

You can pass your own parameter values; below we just need pars/types provided.
"""
function initialize_model(types::TypeParams,
                          parU::UnskilledParams,
                          parS::SkilledParams;
                          Nx::Int = 41,
                          Np::Int = 41,
                          Ny::Int = 41,
                          θ0::Real = 1.0,
                          pstar0::Real = 0.2,
                          u_rate0::Real = 0.1,
                          κ0::Real = 0.5,
                          mS_rate0::Real = 0.05)

    # --- grids (include endpoints exactly) ---
    x = collect(range(0.0, 1.0; length=Nx))
    p = collect(range(0.0, 1.0; length=Np))
    y = collect(range(0.0, 1.0; length=Ny))

    x_edges = bin_edges_from_nodes(x)
    p_edges = p_edges_from_nodes(p)
    y_edges = bin_edges_from_nodes(y)

    gridU = UnskilledGrids(; x=x, p=p, x_edges=x_edges)
    gridS = SkilledGrids(; x=x, y=y, x_edges=x_edges, y_edges=y_edges)

    # --- initialize blocks ---
    u_state, u_cache = initialize_unskilled(parU, types, gridU; θ0=θ0, pstar0=pstar0, u_rate0=u_rate0)
    s_state, s_cache = initialize_skilled(parS, types, gridS; κ0=κ0, mS_rate0=mS_rate0)

    return gridU, gridS, u_state, u_cache, s_state, s_cache
end

initialize_model

In [13]:
# =========================
# Cell — Quick test: manual parameters → initialize_model → solve_full_model!
# =========================

# --- 1) Manual parameter values (tweak freely) ---
types = TypeParams(;
    a_x = 2.0, b_x = 5.0,
    a_y = 2.0, b_y = 2.0
)

parU = UnskilledParams(;
    r  = 0.05,
    ν  = 0.02,

    bU = 0.01,
    bT = 0.08,
    PU = 1.10,

    μU = 1.60,
    ηU = 0.60,

    kU = 1.15,

    λU = 0.05,
    αU = 1.00,

    βU = 0.50,

    ϕ  = 0.03,

    ac = 3.70
)

parS = SkilledParams(;
    r  = parU.r,
    ν  = parU.ν,

    bS = 0.01,
    PS = 1.50,

    μS = 0.45,
    ηS = 0.75,

    s  = 0.10,

    ξS = 0.01,
    δS = 0.01,

    βS = 0.40,

    kS_const = 1.62,

    ω  = 0.50,
    ρ  = 0.61
)

set = Settings(;
    maxit_inner = 1_000,
    maxit_outer = 700,
    tol_inner   = 1e-4,
    tol_outer   = 1e-6,
    verbose     = true
)

# --- 2) Initialize grids + states + caches ---
gridU, gridS, u_state, u_cache, s_state, s_cache =
    initialize_model(types, parU, parS;
        Nx=200, Np=100, Ny=120,
        θ0=1.0, pstar0=0.30, u_rate0=0.12,
        κ0=0.40, mS_rate0=0.05
    )

# --- 3) Solve full model (global fixed point) ---
global_it = solve_full_model!(u_state, u_cache, s_state, s_cache,
                              parU, gridU, parS, gridS;
                              set=set,
                              maxit_global=200,
                              tol_global=5e-7,
                              damp_mS=0.5,
                              damp_US=0.5)

println("\n=== Done ===")
println("global iterations = ", global_it)
@printf("Unskilled: θU = %.6f, avg p* = %.4f, total u = %.4f, total t = %.4f\n",
        u_state.θU, mean(u_state.pstar), sum(u_state.u_mass), sum(u_state.t_mass))
@printf("Skilled:   κ  = %.6f, total mS = %.4f, total uS = %.4f, total eS = %.4f, total vS = %.4f\n",
        s_state.κ, sum(s_state.mS_mass), sum(s_state.uS_mass), sum(s_state.eS_mass), sum(s_state.vS_mass))

# --- 4) Quick sanity checks (should be nonnegative; accounting should hold approximately) ---
@printf("Skilled accounting max|mS - (uS+eS)| = %.3e\n",
        maximum(abs.(s_state.mS_mass .- (s_state.uS_mass .+ s_state.eS_mass))))

@printf("Share feasible in skilled S(x,y): %.2f%%\n",
        100 * mean(Float64.(s_state.feasible)))

# Optional: peek at a couple of objects
println("\nTop-line diagnostics:")
println("min/max p*  = ", (minimum(u_state.pstar), maximum(u_state.pstar)))
println("min/max S   = ", (minimum(s_state.S), maximum(s_state.S)))

inner it=1  maxΔ=5.575e+00  (ΔUsearch=5.575e+00, ΔU=0.000e+00)
inner it=25  maxΔ=2.236e-03  (ΔUsearch=2.143e-03, ΔU=2.236e-03)
inner it=50  maxΔ=7.667e-04  (ΔUsearch=7.346e-04, ΔU=7.667e-04)
inner it=75  maxΔ=2.629e-04  (ΔUsearch=2.519e-04, ΔU=2.629e-04)
inner converged in it=98
outer it=1  maxΔ=4.999e-01  (Δθ=4.999e-01, Δp*=3.500e-01, Δu=1.218e-03)  θ=0.500106
inner it=1  maxΔ=1.130e-02  (ΔUsearch=1.130e-02, ΔU=9.406e-05)
inner converged in it=16
inner it=1  maxΔ=2.537e-01  (ΔUsearch=2.537e-01, ΔU=5.882e-05)
inner it=25  maxΔ=8.178e-05  (ΔUsearch=5.765e-05, ΔU=8.178e-05)
inner converged in it=25
inner it=1  maxΔ=3.509e-01  (ΔUsearch=3.509e-01, ΔU=5.765e-05)
inner it=25  maxΔ=7.575e-05  (ΔUsearch=5.248e-05, ΔU=7.575e-05)
inner converged in it=25
inner it=1  maxΔ=3.985e-01  (ΔUsearch=3.985e-01, ΔU=5.248e-05)
inner converged in it=24
inner it=1  maxΔ=3.774e-01  (ΔUsearch=3.774e-01, ΔU=4.651e-05)
inner converged in it=22
inner it=1  maxΔ=2.237e-01  (ΔUsearch=2.237e-01, ΔU=5.425e-05)
inner

Excessive output truncated after 524360 bytes.

InterruptException: Error trying to display an error.

In [14]:
# -------------------------
# 5) Extra diagnostics: unemployment rates + training take-up τ + implied cutoff
# -------------------------

# Unskilled masses
Uu = sum(u_state.u_mass)
Tu = sum(u_state.t_mass)

# Skilled masses
Us = sum(s_state.uS_mass)
Es = sum(s_state.eS_mass)
Ms = sum(s_state.mS_mass)   # should equal Us + Es

# Market-specific unemployment rates
# (A) "labor force" excludes trainees (treat training as out of labor force)
LF_U_exclT = Uu                      # if unskilled labor force is just active searchers u
LF_S       = Us + Es
UR_U_exclT = (LF_U_exclT > 0) ? Uu / LF_U_exclT : NaN   # will be 1.0 by construction if LF_U_exclT == Uu
UR_S       = (LF_S > 0)       ? Us / LF_S       : NaN

# (B) Alternative: count trainees as in the unskilled labor force (nonstandard, but sometimes you may want it)
LF_U_inclT = Uu + Tu
UR_U_inclT = (LF_U_inclT > 0) ? Uu / LF_U_inclT : NaN

# Global unemployment (two conventions)
N_exclT = LF_U_exclT + LF_S
UR_global_exclT = (N_exclT > 0) ? (Uu + Us) / N_exclT : NaN

N_inclT = LF_U_inclT + LF_S
UR_global_inclT = (N_inclT > 0) ? (Uu + Us) / N_inclT : NaN

println("\n--- Unemployment rates ---")
@printf("Unskilled UR (incl trainees in LF): %.4f   [u/(u+t)]\n", UR_U_inclT)
@printf("Skilled   UR:                      %.4f   [uS/(uS+eS)]\n", UR_S)
@printf("Global UR (incl trainees in LF):   %.4f   [(u+uS)/(u+t+uS+eS)]\n", UR_global_inclT)

println("\n(Alt convention: treat trainees as out of LF)")
@printf("Global UR (excl trainees):         %.4f   [(u+uS)/(uS+eS+u)]\n", UR_global_exclT)

# -------------------------
# Training take-up τ and cutoff proxy
# -------------------------
# τ(x) is the mass share training at each x: t(x)/(u(x)+t(x)).
# Then define a cutoff as the smallest x at which τ(x) exceeds a small threshold.
τ = similar(u_state.u_mass)
@inbounds for ix in eachindex(u_state.u_mass)
    denom = u_state.u_mass[ix] + u_state.t_mass[ix]
    τ[ix] = denom > 0 ? u_state.t_mass[ix] / denom : 0.0
end

τ_bar = sum(u_state.t_mass) / (sum(u_state.t_mass) + sum(u_state.u_mass) + 1e-16)

# implied cutoff: first grid point where take-up exceeds 50% (and also report 10% as robustness)
function implied_cutoff(xgrid, τvec; thr=0.5)
    for i in eachindex(τvec)
        if τvec[i] >= thr
            return xgrid[i]
        end
    end
    return NaN
end

xcut50 = implied_cutoff(gridU.x, τ; thr=0.5)
xcut10 = implied_cutoff(gridU.x, τ; thr=0.1)

println("\n--- Training take-up τ ---")
@printf("Average take-up τ̄ (mass-weighted): %.4f\n", τ_bar)
@printf("Implied cutoff x where τ(x)≥10%%:   %.4f\n", xcut10)
@printf("Implied cutoff x where τ(x)≥50%%:   %.4f\n", xcut50)
@printf("min/max τ(x):                      (%.4f, %.4f)\n", minimum(τ), maximum(τ))

# Optional: show "who trains" in terms of average x among trainees vs non-trainees
# (These are crude moments using masses, not the underlying type density.)
x = gridU.x
mass_t = sum(u_state.t_mass) + 1e-16
mass_u = sum(u_state.u_mass) + 1e-16
xbar_t = sum(x .* u_state.t_mass) / mass_t
xbar_u = sum(x .* u_state.u_mass) / mass_u

@printf("Mean x among trainees:             %.4f\n", xbar_t)
@printf("Mean x among non-trainees (u):     %.4f\n", xbar_u)

In [ ]:
# =========================
# Cell — Plots (unskilled + skilled)
# =========================
using Plots

# --- helpers ---
function _safe01(y)
    # avoid weird negative numerical dust in densities
    return max.(y, 0.0)
end

function plot_unskilled(u_state::UnskilledState, gridU::UnskilledGrids; title_prefix="Unskilled")
    x = gridU.x
    p = gridU.p

    # τ as {0,1}
    τf = Float64.(u_state.τ)

    plt1 = plot(x, u_state.Usearch; xlabel="x", ylabel="value", title="$(title_prefix): Usearch / U / T", label="Usearch")
    plot!(plt1, x, u_state.U;     label="U")
    plot!(plt1, x, u_state.Tval;  label="T")

    plt2 = plot(x, u_state.pstar; xlabel="x", ylabel="p*(x)", title="$(title_prefix): reservation p*(x)", label="p*")
    hline!(plt2, [0.0, 1.0]; label=["" ""], linestyle=:dash)

    plt3 = plot(x, τf; xlabel="x", ylabel="τ(x)", title="$(title_prefix): training policy", label="τ", ylim=(-0.05, 1.05))

    plt4 = plot(x, _safe01(u_state.u_mass); xlabel="x-bin node", ylabel="mass", title="$(title_prefix): stationary masses by x-bin", label="u_mass")
    plot!(plt4, x, _safe01(u_state.t_mass); label="t_mass")

    # Heatmaps for JU and EU (x,p)
    plt5 = heatmap(p, x, u_state.JU; xlabel="p", ylabel="x", title="$(title_prefix): J_U(x,p)", colorbar_title="J")
    plt6 = heatmap(p, x, u_state.EU; xlabel="p", ylabel="x", title="$(title_prefix): E_U(x,p)", colorbar_title="E")

    display(plt1); display(plt2); display(plt3); display(plt4); display(plt5); display(plt6)
    return nothing
end

function plot_skilled(s_state::SkilledState, gridS::SkilledGrids; title_prefix="Skilled")
    x = gridS.x
    y = gridS.y

    plt1 = plot(x, _safe01(s_state.mS_mass); xlabel="x", ylabel="mass", title="$(title_prefix): trained mass accounting", label="mS")
    plot!(plt1, x, _safe01(s_state.uS_mass); label="uS")
    plot!(plt1, x, _safe01(s_state.eS_mass); label="eS")

    plt2 = plot(y, _safe01(s_state.vS_mass); xlabel="y", ylabel="mass", title="$(title_prefix): vacancy mass by y-bin", label="vS")

    # Surplus and matches heatmaps
    plt3 = heatmap(y, x, s_state.S; xlabel="y", ylabel="x", title="$(title_prefix): surplus S(x,y)", colorbar_title="S")
    plt4 = heatmap(y, x, s_state.h_mass; xlabel="y", ylabel="x", title="$(title_prefix): match mass h(x,y)", colorbar_title="h")

    display(plt1); display(plt2); display(plt3); display(plt4)
    return nothing
end

# --- run plots (assumes you already ran the "Quick test" cell and have gridU, gridS, u_state, s_state) ---
plot_unskilled(u_state, gridU)
plot_skilled(s_state, gridS)

# Optional: compact summary plot for skilled feasibility boundary (share feasible by x)
feas_share_by_x = vec(mean(Float64.(s_state.feasible), dims=2))
plt_feas = plot(gridS.x, feas_share_by_x; xlabel="x", ylabel="share feasible across y",
                title="Skilled: feasibility share by worker type x", label="E_y[1{S≥0}]")
display(plt_feas)